# Data Quality Audit of Raw Data from the ENTSO-E Transparency Platform

The main objective of the data quality audit is to assess the quality, consistency, and general characteristics of the raw data obtained from the ENTSO-E Transparency Platform.

The analysis focuses on the following aspects:
* Missing data and missing observations
* Index integrity and consistency
* Duplicated indices
* Timestamp and time zone consistency
* Overall data characteristics

**All data and variables fetched from ENTSO-E are explained in the [ENTSO-E data overview](./01_ENTSOE_data_overview.ipynb).**

In [ ]:
import sys
from pathlib import Path

import pandas as pd

sys.path.append(str(Path.cwd().parent.parent.parent))
from src.config import (
    DATA_DIR,
    DOWNLOAD_START_DATE,
    DOWNLOAD_END_DATE,
    ENTSOE_BIDDING_ZONE_CODES,
)
from src.utils.data_quality import (
    print_index_summary,
    print_missing_values,
    print_data_overview,
    find_missing_dates,
    print_units_unavailability_analysis,
)

from src.utils.file_io import load_entsoe_data

In [2]:
data_dir = DATA_DIR / "raw" / "entsoe"
date_range = f"{DOWNLOAD_START_DATE}_{DOWNLOAD_END_DATE}"
zone_codes = ENTSOE_BIDDING_ZONE_CODES

In [3]:
data_entsoe = load_entsoe_data(
    DATA_DIR / "raw" / "entsoe",
    date_range,
)

## 1. Poland (PL) data

### 1.1 Missing Data

In [4]:
data_poland = data_entsoe["PL"]
for name, data in data_poland.items():
    print_missing_values(data=data, name=name)


--- [load] Missing values ---
No missing values.

--- [generation] Missing values ---
Other renewable                              13529
Other                                        13529
Hydro Pumped Storage - Actual Consumption    32829
Wind Offshore                                85721

--- [load_forecast] Missing values ---
No missing values.

--- [day_ahead_prices] Missing values ---
No missing values.

--- [generation_wind_solar_forecast] Missing values ---
Wind Offshore    83993

--- [generation_units_unavailability] Missing values ---
docstatus        31365
nominal_power       29

--- [production_units_unavailability] Missing values ---
docstatus    1221

--- [generation_forecast] Missing values ---
No missing values.


**Generation** 

In [5]:
missing_generation_types = [
    "Other renewable",
    "Other",
    "Hydro Pumped Storage - Actual Consumption",
    "Wind Offshore",
]

poland_generation = data_poland["generation"]

poland_missing_generation = poland_generation[missing_generation_types].copy()

poland_missing_generation["total_generation"] = poland_generation.sum(axis=1)

poland_missing_generation["missing_generation_total"] = poland_missing_generation[
    missing_generation_types
].sum(axis=1)

poland_missing_generation["missing_generation_share_pct"] = (
    poland_missing_generation["missing_generation_total"]
    / poland_missing_generation["total_generation"]
).round(2) * 100

print(
    f"Mean share of total generation from missing generation types: "
    f'{poland_missing_generation["missing_generation_share_pct"].mean().round(2)}%'
)
poland_missing_generation

Mean share of total generation from missing generation types: 3.74%


,Other renewable,Other,Hydro Pumped Storage - Actual Consumption,Wind Offshore,total_generation,missing_generation_total,missing_generation_share_pct
2022-12-01 00:00:00+01:00,NaN,NaN,NaN,NaN,18124.850,0.000,0.0
2022-12-01 01:00:00+01:00,NaN,NaN,NaN,NaN,17878.200,0.000,0.0
2022-12-01 02:00:00+01:00,NaN,NaN,NaN,NaN,17780.410,0.000,0.0
2022-12-01 03:00:00+01:00,NaN,NaN,NaN,NaN,17754.070,0.000,0.0
2022-12-01 04:00:00+01:00,NaN,NaN,NaN,NaN,18131.320,0.000,0.0
...,...,...,...,...,...,...,...
2026-09-15 22:45:00+02:00,54.998,552.439,6.737,87.6,17387.193,701.774,4.0
2026-09-15 23:00:00+02:00,54.492,547.993,8.339,110.4,17564.268,721.224,4.0
2026-09-15 23:15:00+02:00,54.155,544.366,8.339,134.8,17418.495,741.660,4.0
2026-09-15 23:30:00+02:00,54.048,544.348,12.090,156.8,17379.289,767.286,4.0


### 1.2 Missing Data – Observations

Missing values in the core series occur only in `generation` and `generation_wind_solar_forecast`. Most gaps are structural (a series not reported yet), not random loss.

`Other`, `Other renewable` and `Hydro Pumped Storage - Actual Consumption` are NaN at the start of the sample and populated later; `Wind Offshore` has values only at the end (first offshore sources went live in July 2026). Combined, these four columns contribute on average only 3.74% of total generation (over the whole sample), confirming they can be dropped without a significant loss of signal.

In `generation_wind_solar_forecast`, only `Wind Offshore` is missing, and its forecast series starts earlier than the actual offshore output.

### 1.3 Index Integrity

In [6]:
for name, data in data_poland.items():
    print_index_summary(data=data, name=name)


--- [load] Index summary ---
Duplicates: 0
Duplicates due to Summer/Winter time switch: 9
Timezone:   Europe/Warsaw

Time steps:
  0 days 00:15:00: 79191
  0 days 01:00:00: 13441

--- [generation] Index summary ---
Duplicates: 0
Duplicates due to Summer/Winter time switch: 9
Timezone:   Europe/Warsaw

Time steps:
  0 days 00:15:00: 79191
  0 days 01:00:00: 13441

--- [load_forecast] Index summary ---
Duplicates: 0
Duplicates due to Summer/Winter time switch: 9
Timezone:   Europe/Warsaw

Time steps:
  0 days 00:15:00: 79191
  0 days 01:00:00: 13441

--- [day_ahead_prices] Index summary ---
Duplicates: 45
Duplicates due to Summer/Winter time switch: 6
Timezone:   Europe/Warsaw

Time steps:
  0 days 00:15:00: 33600
  0 days 01:00:00: 24839
  0 days 00:00:00: 45

--- [generation_wind_solar_forecast] Index summary ---
Duplicates: 0
Duplicates due to Summer/Winter time switch: 9
Timezone:   Europe/Warsaw

Time steps:
  0 days 00:15:00: 79191
  0 days 01:00:00: 13441

--- [generation_units_u

**Day-ahead prices**

In [7]:
poland_day_ahead_prices = data_poland["day_ahead_prices"].copy()
duplicated_poland_day_ahead_prices = poland_day_ahead_prices[
    poland_day_ahead_prices.index.duplicated(keep=False)
]
duplicated_poland_day_ahead_prices

,day_ahead_prices
2023-01-01 00:00:00+01:00,18.09
2023-01-01 00:00:00+01:00,18.09
2023-02-01 00:00:00+01:00,85.00
2023-02-01 00:00:00+01:00,85.00
2023-03-01 00:00:00+01:00,134.13
...,...
2026-07-01 00:00:00+02:00,151.12
2026-08-01 00:00:00+02:00,184.07
2026-08-01 00:00:00+02:00,184.07
2026-09-01 00:00:00+02:00,172.41


**Generation/Production units unavailability**

In [8]:
units_clean = {}

print("Poland - generation_units_unavailability")
units_clean["PL", "generation_units_unavailability"] = (
    print_units_unavailability_analysis(
        data=data_poland["generation_units_unavailability"]
    )
)
units_clean["PL", "generation_units_unavailability"]

Poland - generation_units_unavailability
Initial number of rows: 31365
Duplicated rows: 1665
Duplicated (mrid, revision) rows: 0

Rows with created_doc_time more than 3 years before start:
businesstype
Planned maintenance    17
Name: count, dtype: int64

Rows with created_doc_time on or after end:
businesstype
Planned maintenance    10420
Unplanned outage        2897
Name: count, dtype: int64

Final number of rows: 16366

Planned maintenance:
Number of rows: 15242
created_doc_time range: 2017-11-30 12:47:38+00:00 - 2026-09-30 10:38:46+00:00

Unplanned outage:
Number of rows: 1124
created_doc_time range: 2019-12-05 10:05:27+00:00 - 2026-09-17 21:24:21+00:00


,created_doc_time,avail_qty,biddingzone_domain,businesstype,curvetype,docstatus,end,mrid,nominal_power,plant_type,production_resource_id,production_resource_location,production_resource_name,production_resource_psr_name,pstn,qty_uom,resolution,revision,start
0,2017-11-30 13:47:38+01:00,0,PL,Planned maintenance,A03,None,2019-12-09 00:00:00+01:00,CP9MmH3Mbq67eltC7kIFAg,390.0,Fossil Brown coal/Lignite,19W000000000016F,Polska,Bełchatów,Bełchatów B07,1,MAW,PT1M,8,2019-09-09 00:00:00+02:00
1,2017-11-30 13:47:38+01:00,0,PL,Planned maintenance,A03,None,2019-12-16 00:00:00+01:00,8meeC2IibBbK_4t2j1j23A,390.0,Fossil Brown coal/Lignite,19W000000000016F,Polska,Bełchatów,Bełchatów B08,1,MAW,PT1M,9,2019-09-16 00:00:00+02:00
2,2017-11-30 13:47:46+01:00,0,PL,Planned maintenance,A03,None,2020-01-01 00:00:00+01:00,ad09axfZQy3j9Xh2THvIfw,222.0,Fossil Hard coal,19W000000000043C,Polska,Dolna Odra,Dolna Odra B6,1,MAW,PT1M,4,2019-09-02 00:00:00+02:00
3,2017-11-30 13:48:39+01:00,0,PL,Planned maintenance,A03,None,2020-03-30 00:00:00+02:00,HmUdfCUi0b2S2MmBTOr7uQ,225.0,Fossil Hard coal,19W0000000000636,Polska,Jaworzno 3,Jaworzno 3 B4,1,MAW,PT1M,6,2019-11-01 00:00:00+01:00
4,2017-11-30 13:48:40+01:00,0,PL,Planned maintenance,A03,None,2020-03-30 00:00:00+02:00,xvreXhJWcXYp6zt23kiHAQ,225.0,Fossil Hard coal,19W0000000000636,Polska,Jaworzno 3,Jaworzno 3 B3,1,MAW,PT1M,9,2019-11-01 00:00:00+01:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29694,2026-09-24 10:31:20+02:00,0,PL,Planned maintenance,A03,None,2026-09-28 00:00:00+02:00,yfwejCTZis8jNUQgIK1BEA,834.0,Fossil Hard coal,19W000000000296K,Polska,Jaworzno 2 JWCD,Jaworzno 2 B7,1,MAW,PT1M,4,2026-07-06 00:04:00+02:00
29696,2026-09-26 13:12:03+02:00,0,PL,Planned maintenance,A03,None,2026-09-27 00:00:00+02:00,Mztc_GNS8kjyDkNcJOgfbw,213.0,Fossil Hard coal,19W000000000095U,Polska,Kozienice 1,Kozienice 1 B4,1,MAW,PT1M,6,2026-09-09 00:14:00+02:00
29697,2026-09-28 11:53:11+02:00,0,PL,Planned maintenance,A03,None,2026-10-08 00:00:00+02:00,y7p5VuV_tUV17rJtt9w8ew,358.0,Fossil Hard coal,19W0000000001292,Polska,Opole,Opole B2,1,MAW,PT1M,3,2026-09-15 03:25:00+02:00
29698,2026-09-29 10:42:52+02:00,0,PL,Planned maintenance,A03,None,2026-10-09 22:00:00+02:00,1leTtfDpmZLdE8NouKtBbQ,203.0,Fossil Hard coal,19W000000000095U,Polska,Kozienice 1,Kozienice 1 B7,1,MAW,PT1M,7,2026-09-06 08:17:00+02:00


In [9]:
print("Poland - production_units_unavailability")
units_clean["PL", "production_units_unavailability"] = (
    print_units_unavailability_analysis(
        data=data_poland["production_units_unavailability"]
    )
)
units_clean["PL", "production_units_unavailability"]

Poland - production_units_unavailability
Initial number of rows: 1221
Duplicated rows: 96
Duplicated (mrid, revision) rows: 0

Rows with created_doc_time more than 3 years before start:
businesstype
Planned maintenance    2
Name: count, dtype: int64

Rows with created_doc_time on or after end:
businesstype
Unplanned outage       603
Planned maintenance    388
Name: count, dtype: int64

Final number of rows: 132

Planned maintenance:
Number of rows: 90
created_doc_time range: 2017-11-30 12:52:13+00:00 - 2025-10-06 23:28:26+00:00

Unplanned outage:
Number of rows: 42
created_doc_time range: 2019-12-08 11:35:19+00:00 - 2022-05-26 07:51:42+00:00


,created_doc_time,avail_qty,biddingzone_domain,businesstype,curvetype,docstatus,end,mrid,nominal_power,plant_type,production_resource_id,production_resource_location,production_resource_name,production_resource_psr_name,pstn,qty_uom,resolution,revision,start
1,2017-11-30 13:52:13+01:00,959,PL,Planned maintenance,A03,None,2020-08-21 00:00:00+02:00,oh-QXFJcoiz9YHP-7Cle0g,189.0,Fossil Hard coal,19W0000000000628,Polska,Jaworzno 2,,1,MAW,PT1M,2,2020-07-04 00:00:00+02:00
2,2018-12-03 14:00:10+01:00,110.5,PL,Planned maintenance,A03,None,2021-06-12 00:00:00+02:00,Mlr7CV5HKPm8fMBaP2ZSww,217.3,Fossil Hard coal,19W000000000059Y,Polska,Gdańsk 2,,1,MAW,PT1M,1,2021-05-28 00:00:00+02:00
3,2019-11-29 12:29:16+01:00,95,PL,Planned maintenance,A03,None,2020-08-17 00:00:00+02:00,cQBFPtWHTeewoyh8PxxTcg,205.0,Fossil Hard coal,19W000000000106E,Polska,EC Łódź-3,,1,MAW,PT1M,1,2020-08-01 00:00:00+02:00
4,2019-11-29 12:29:38+01:00,111.5,PL,Planned maintenance,A03,None,2020-09-15 00:00:00+02:00,Gb_7I3h3pJ2onUa-i_nBvA,217.3,Fossil Hard coal,19W000000000059Y,Polska,Gdańsk 2,,1,MAW,PT1M,1,2020-09-01 00:00:00+02:00
5,2019-11-29 12:31:53+01:00,229,PL,Planned maintenance,A03,None,2020-06-26 00:00:00+02:00,-D4r8GtMqaH6tr0JbA1A9Q,122.0,Fossil Hard coal,19W000000000229Z,Polska,EC Żerań,,1,MAW,PT1M,1,2020-06-19 00:00:00+02:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
249,2022-06-09 12:40:26+02:00,132,PL,Planned maintenance,A03,None,2022-06-10 00:00:00+02:00,iWBMu6bEnuRaTTr2mC-U-g,299.0,Fossil Hard coal,19W000000000297I,Polska,EC Żerań 1,,1,MAW,PT1M,2,2022-05-24 15:01:00+02:00
251,2022-06-20 13:13:34+02:00,105,PL,Planned maintenance,A03,None,2022-06-21 15:00:00+02:00,MxvUy4spe6095pDX9XCQRg,299.0,Fossil Hard coal,19W000000000297I,Polska,EC Żerań 1,,1,MAW,PT1M,3,2022-06-18 01:31:00+02:00
382,2025-10-07 01:28:19+02:00,505,PL,Planned maintenance,A03,None,2026-08-18 00:00:00+02:00,5az3Te-5EIACvlWo89amXg,533.8,Fossil Hard coal,19W0000000002191,Polska,EC Siekierki,,1,MAW,PT1M,1,2026-06-29 00:01:00+02:00
383,2025-10-07 01:28:19+02:00,473,PL,Planned maintenance,A03,None,2026-06-29 00:00:00+02:00,8fQR97ll9FqGiLr8G_QZxw,533.8,Fossil Hard coal,19W0000000002191,Polska,EC Siekierki,,1,MAW,PT1M,2,2026-06-08 00:01:00+02:00


**Unavailability records by outage start month (after deduplication and filtering)**

In [10]:
for name in ["generation_units_unavailability", "production_units_unavailability"]:
    start = units_clean["PL", name]["start"]
    start = start[start >= "2023-01-01"]
    print(f"Poland - {name} (number of records by outage start month)")
    print(
        pd.crosstab(
            index=start.dt.year.to_numpy(),
            columns=start.dt.month.to_numpy(),
            rownames=["year"],
            colnames=["month"],
        ),
        end="\n\n",
    )

Poland - generation_units_unavailability (number of records by outage start month)
month   1     2     3     4    5    6    7    8    9   10   11   12
year                                                               
2024     0     0     0     0    0    1    1    0    0   1    0    0
2025     0     0     0     0    1    1    2    0    0   3  532  548
2026   708  1005  1066  1046  811  917  818  816  359   0    0    0

Poland - production_units_unavailability (number of records by outage start month)
month  6  7  8  9
year             
2023   0  1  2  1
2026   2  1  0  0



### 1.4 Index Integrity – Observations

All PL series are tz-aware (Europe/Warsaw), no timestamps are missing. Resolution is mixed: `load`, `generation` and the forecast tables share one index (13,441 hourly + 79,191 15-min steps); `day_ahead_prices` stayed hourly much longer (24,839 hourly steps), so the 2025 MTU change affects only prices. A common grid is needed before merging.

DST repeats (9 timestamps, 6 in prices) are valid distinct instants, not duplicates. `day_ahead_prices` has 45 genuine exact duplicates, always at 00:00 on the first day of a month, and should be deduplicated.

**Unavailability tables** (`generation_units_unavailability`, `production_units_unavailability`). Apart from 1,665 exact duplicates in the generation-unit table (no repeated `(mrid, revision)` pairs), most records are retrospective: 13,317 of 31,365 generation-unit and 991 of 1,221 production-unit records have `created_doc_time >= end`, i.e. they were published only after the outage ended and would not be known at forecast time. After removing duplicates, retrospective records and records with `created_doc_time` more than 3 years before `start`, 16,366 generation-unit and 132 production-unit records remain.

The monthly counts above show that the remaining generation-unit records appear only at the end of the sample: none in 2023, at most 1 per month in 2024 and at most 2 per month until September 2025, then 532 in November and 548 in December 2025 and 359–1,066 per month in 2026. `production_units_unavailability` has only 7 records starting from 2023.

**Decision:** both tables are excluded from the dataset. Informative history exists only for the last months of the sample, so the training (to 2024-12-29) and validation (to 2025-08-31) periods would contain practically no data, and keeping the retrospective records would use information unavailable at forecast time.

### 1.5 Data Descriptive Statistics

In [11]:
for name, data in data_poland.items():
    print_data_overview(data=data, name=name)


--- [load] Overview ---
Shape: 92633 x 1

Info:
<class 'pandas.DataFrame'>
DatetimeIndex: 92633 entries, 2022-12-01 00:00:00+01:00 to 2026-09-15 23:45:00+02:00
Data columns (total 1 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Actual Load  92633 non-null  float64
dtypes: float64(1)
memory usage: 1.4 MB

Description:
        Actual Load
count  92633.000000
mean   18305.505318
std     3152.694048
min    10463.962000
25%    15753.941000
50%    18411.141000
75%    20451.363000
max    28303.880000

--- [generation] Overview ---
Shape: 92633 x 15

Info:
<class 'pandas.DataFrame'>
DatetimeIndex: 92633 entries, 2022-12-01 00:00:00+01:00 to 2026-09-15 23:45:00+02:00
Data columns (total 15 columns):
 #   Column                                     Non-Null Count  Dtype  
---  ------                                     --------------  -----  
 0   Biomass                                    92633 non-null  float64
 1   Fossil Brown coal/Lignite

            Biomass  Fossil Brown coal/Lignite  Fossil Coal-derived gas  \
count  92633.000000               92633.000000             92633.000000   
mean     303.949663                3589.136399               149.397911   
std       86.161133                1072.705949                43.108341   
min        0.000000                 902.873000                 9.000000   
25%      255.740000                2763.174000               136.574000   
50%      313.740000                3637.664000               160.791000   
75%      369.897000                4435.401000               180.221000   
max      454.637000                6557.615000               226.894000   

         Fossil Gas  Fossil Hard coal    Fossil Oil  Hydro Pumped Storage  \
count  92633.000000      92633.000000  92633.000000          92633.000000   
mean    2182.963557       6289.817030    185.140519            130.509217   
std      809.232546       1836.949925     62.921216            243.509467   
min      247.230

## 2. Other European Bidding Zones included data

In [12]:
data_entose = {bz: dataset for bz, dataset in data_entsoe.items() if bz != "PL"}

### 2.1 Missing Data

In [13]:
for bz, datasets_bz in data_entose.items():
    print(f"\n--- {bz} " + "-" * 40)
    for name, data in datasets_bz.items():
        print_missing_values(data=data, name=name)


--- AT ----------------------------------------

--- [load_forecast] Missing values ---
No missing values.

--- [water_reservoirs_hydro_storage] Missing values ---
No missing values.

--- [day_ahead_prices] Missing values ---
No missing values.

--- [generation_wind_solar_forecast] Missing values ---
No missing values.

--- [generation_forecast] Missing values ---
No missing values.

--- CZ ----------------------------------------

--- [load_forecast] Missing values ---
No missing values.

--- [day_ahead_prices] Missing values ---
No missing values.

--- [generation_wind_solar_forecast] Missing values ---
No missing values.

--- [generation_forecast] Missing values ---
No missing values.

--- DE_LU ----------------------------------------

--- [load_forecast] Missing values ---
No missing values.

--- [day_ahead_prices] Missing values ---
No missing values.

--- [generation_wind_solar_forecast] Missing values ---
Solar           12
Wind Onshore    12

--- [generation_units_unavailabil

**DE_LU - generation wind and solar forecast**

In [14]:
print("[DE_LU] - missing values for generation_wind_solar_forecast\n")

missing_wind_onshore_de_lu = data_entose["DE_LU"]["generation_wind_solar_forecast"][
    "Wind Onshore"
][data_entose["DE_LU"]["generation_wind_solar_forecast"]["Wind Onshore"].isna()]
print("\nWind Onshore")
print(f"total missing: {len(missing_wind_onshore_de_lu)}\n")
print(missing_wind_onshore_de_lu.index)

missing_solar_de_lu = data_entose["DE_LU"]["generation_wind_solar_forecast"]["Solar"][
    data_entose["DE_LU"]["generation_wind_solar_forecast"]["Solar"].isna()
]
print("\nSolar")
print(f"total missing: {len(missing_solar_de_lu)}\n")
print(missing_solar_de_lu.index)

[DE_LU] - missing values for generation_wind_solar_forecast


Wind Onshore
total missing: 12

DatetimeIndex(['2023-10-29 00:00:00+02:00', '2023-10-29 00:15:00+02:00',
               '2023-10-29 00:30:00+02:00', '2023-10-29 00:45:00+02:00',
               '2024-10-27 00:00:00+02:00', '2024-10-27 00:15:00+02:00',
               '2024-10-27 00:30:00+02:00', '2024-10-27 00:45:00+02:00',
               '2025-10-26 00:00:00+02:00', '2025-10-26 00:15:00+02:00',
               '2025-10-26 00:30:00+02:00', '2025-10-26 00:45:00+02:00'],
              dtype='datetime64[us, Europe/Berlin]', freq=None)

Solar
total missing: 12

DatetimeIndex(['2023-10-29 00:00:00+02:00', '2023-10-29 00:15:00+02:00',
               '2023-10-29 00:30:00+02:00', '2023-10-29 00:45:00+02:00',
               '2024-10-27 00:00:00+02:00', '2024-10-27 00:15:00+02:00',
               '2024-10-27 00:30:00+02:00', '2024-10-27 00:45:00+02:00',
               '2025-10-26 00:00:00+02:00', '2025-10-26 00:15:00+02:00',
            

**FR - generation wind and solar forecast**

In [15]:
print("[FR] - missing values for generation_wind_solar_forecast\n")

missing_wind_onshore_fr = data_entose["FR"]["generation_wind_solar_forecast"][
    "Wind Onshore"
][data_entose["FR"]["generation_wind_solar_forecast"]["Wind Onshore"].isna()]
print("\nWind Onshore")
print(f"total missing: {len(missing_wind_onshore_fr)}\n")
print(missing_wind_onshore_fr.index)

missing_wind_offshore_fr = data_entose["FR"]["generation_wind_solar_forecast"][
    "Wind Offshore"
][data_entose["FR"]["generation_wind_solar_forecast"]["Wind Offshore"].isna()]
print("\nWind Offshore")
print(f"total missing: {len(missing_wind_offshore_fr)}\n")
print(missing_wind_offshore_fr.index)

missing_solar_fr = data_entose["FR"]["generation_wind_solar_forecast"]["Solar"][
    data_entose["FR"]["generation_wind_solar_forecast"]["Solar"].isna()
]
print("\nSolar")
print(f"total missing: {len(missing_solar_fr)}\n")
print(missing_solar_fr.index)

[FR] - missing values for generation_wind_solar_forecast


Wind Onshore
total missing: 27

DatetimeIndex(['2023-10-29 23:00:00+01:00', '2024-10-27 23:00:00+01:00',
               '2025-10-26 23:00:00+01:00', '2026-01-02 00:00:00+01:00',
               '2026-01-02 01:00:00+01:00', '2026-01-02 02:00:00+01:00',
               '2026-01-02 03:00:00+01:00', '2026-01-02 04:00:00+01:00',
               '2026-01-02 05:00:00+01:00', '2026-01-02 06:00:00+01:00',
               '2026-01-02 07:00:00+01:00', '2026-01-02 08:00:00+01:00',
               '2026-01-02 09:00:00+01:00', '2026-01-02 10:00:00+01:00',
               '2026-01-02 11:00:00+01:00', '2026-01-02 12:00:00+01:00',
               '2026-01-02 13:00:00+01:00', '2026-01-02 14:00:00+01:00',
               '2026-01-02 15:00:00+01:00', '2026-01-02 16:00:00+01:00',
               '2026-01-02 17:00:00+01:00', '2026-01-02 18:00:00+01:00',
               '2026-01-02 19:00:00+01:00', '2026-01-02 20:00:00+01:00',
               '2026-01-02 21:00:

**HU - generation wind and solar forecast**

In [16]:
print("[HU] - missing values generation_wind_solar_forecast")
missing_wind_onshore_hu = data_entose["HU"]["generation_wind_solar_forecast"][
    "Wind Onshore"
][data_entose["HU"]["generation_wind_solar_forecast"]["Wind Onshore"].isna()]
print("\nWind Onshore")
print(f"total missing: {len(missing_wind_onshore_hu)}\n")
print(missing_wind_onshore_hu.index)

[HU] - missing values generation_wind_solar_forecast

Wind Onshore
total missing: 34

DatetimeIndex(['2026-02-16 15:30:00+01:00', '2026-02-16 15:45:00+01:00',
               '2026-02-16 16:00:00+01:00', '2026-02-16 16:15:00+01:00',
               '2026-02-16 16:30:00+01:00', '2026-02-16 16:45:00+01:00',
               '2026-02-16 17:00:00+01:00', '2026-02-16 17:15:00+01:00',
               '2026-02-16 17:30:00+01:00', '2026-02-16 17:45:00+01:00',
               '2026-02-16 18:00:00+01:00', '2026-02-16 18:15:00+01:00',
               '2026-02-16 18:30:00+01:00', '2026-02-16 18:45:00+01:00',
               '2026-02-16 19:00:00+01:00', '2026-02-16 19:15:00+01:00',
               '2026-02-16 19:30:00+01:00', '2026-02-16 19:45:00+01:00',
               '2026-02-16 20:00:00+01:00', '2026-02-16 20:15:00+01:00',
               '2026-02-16 20:30:00+01:00', '2026-02-16 20:45:00+01:00',
               '2026-02-16 21:00:00+01:00', '2026-02-16 21:15:00+01:00',
               '2026-02-16 21:30:00+01

**LT - generation forecast**

In [17]:
print("[LT] - generation forecast data schema")
data_entose["LT"]["generation_forecast"]

[LT] - generation forecast data schema


,Scheduled Consumption,Scheduled Aggregated,Actual Aggregated
2022-12-01 01:00:00+02:00,670.0,278.87,NaN
2022-12-01 02:00:00+02:00,889.0,277.37,NaN
2022-12-01 03:00:00+02:00,886.0,276.47,NaN
2022-12-01 04:00:00+02:00,882.0,276.67,NaN
2022-12-01 05:00:00+02:00,670.0,367.87,NaN
...,...,...,...
2026-09-15 23:45:00+03:00,0.0,290.90,NaN
2026-09-16 00:00:00+03:00,0.0,286.90,NaN
2026-09-16 00:15:00+03:00,0.0,298.50,NaN
2026-09-16 00:30:00+03:00,0.0,310.00,NaN


### 2.2 Missing Data – Observations

DE_LU and FR wind/solar forecast gaps line up with the three autumn DST-change Sundays (DE_LU: first hour; FR: last hour), so they are DST artefacts, not random loss. FR `Wind Offshore` is missing as one block from the sample start to 2023-08-06 — structural, forecasts simply start later. FR `Solar` and `Wind Onshore` are also missing for the whole of 2026-01-02. HU has one isolated 8.5-hour gap in `Wind Onshore` on 2026-02-16.

LT `generation_forecast` uses a different schema (three columns instead of one) and `Actual Aggregated` is mostly empty - not usable in its current form.

### 2.3 Index Integrity

In [18]:
for bz, datasets_bz in data_entose.items():
    print(f"\n--- {bz} " + "-" * 40)
    for name, data in datasets_bz.items():
        print_index_summary(data=data, name=name)


--- AT ----------------------------------------

--- [load_forecast] Index summary ---
Duplicates: 0
Duplicates due to Summer/Winter time switch: 12
Timezone:   Europe/Vienna

Time steps:
  0 days 00:15:00: 132955

--- [water_reservoirs_hydro_storage] Index summary ---
Duplicates: 0
Duplicates due to Summer/Winter time switch: 0
Timezone:   UTC

Time steps:
  7 days 00:00:00: 194
  14 days 00:00:00: 1

--- [day_ahead_prices] Index summary ---
Duplicates: 0
Duplicates due to Summer/Winter time switch: 6
Timezone:   Europe/Vienna

Time steps:
  0 days 00:15:00: 33598
  0 days 01:00:00: 24835
  0 days 02:00:00: 2
  0 days 00:30:00: 1

--- [generation_wind_solar_forecast] Index summary ---
Duplicates: 0
Duplicates due to Summer/Winter time switch: 12
Timezone:   Europe/Vienna

Time steps:
  0 days 00:15:00: 132955

--- [generation_forecast] Index summary ---
Duplicates: 0
Duplicates due to Summer/Winter time switch: 3
Timezone:   Europe/Vienna

Time steps:
  0 days 00:15:00: 28795
  0 day

**DE_LU - generation/production units unavailability**

In [19]:
units_clean["DE_LU", "generation_units_unavailability"] = (
    print_units_unavailability_analysis(
        data=data_entose["DE_LU"]["generation_units_unavailability"]
    )
)
units_clean["DE_LU", "generation_units_unavailability"]

Initial number of rows: 116263
Duplicated rows: 42272
Duplicated (mrid, revision) rows: 38744

Rows with created_doc_time more than 3 years before start:
businesstype
Planned maintenance    37
Name: count, dtype: int64

Rows with created_doc_time on or after end:
businesstype
Planned maintenance    16406
Unplanned outage        8918
Name: count, dtype: int64

Final number of rows: 9886

Planned maintenance:
Number of rows: 7736
created_doc_time range: 2017-07-26 12:14:00+00:00 - 2026-10-01 12:54:24+00:00

Unplanned outage:
Number of rows: 2150
created_doc_time range: 2019-05-29 10:54:41+00:00 - 2026-10-01 08:45:41+00:00


,created_doc_time,avail_qty,biddingzone_domain,businesstype,curvetype,docstatus,end,mrid,nominal_power,plant_type,production_resource_id,production_resource_location,production_resource_name,production_resource_psr_name,pstn,qty_uom,resolution,revision,start
0,2017-07-26 14:14:00+02:00,140,DE_LU,Planned maintenance,A03,None,2020-06-23 00:00:00+02:00,N6eNDtYkDL64plOlhkRfEw,111.2,Fossil Hard coal,11WD4GKM-XCD0--3,intra_zonal,GKM AG DBEnergie,GKM AG DBEnergie,1,MAW,PT1M,4,2020-06-13 00:00:00+02:00
1,2017-07-26 14:14:00+02:00,109,DE_LU,Planned maintenance,A03,None,2020-06-13 00:00:00+02:00,-W9tHcAygE5cWAQ7-duUzw,111.2,Fossil Hard coal,11WD4GKM-XCD0--3,intra_zonal,GKM AG DBEnergie,GKM AG DBEnergie,1,MAW,PT1M,4,2020-06-01 00:00:00+02:00
2,2017-07-26 14:14:02+02:00,397,DE_LU,Planned maintenance,A03,None,2020-06-23 00:00:00+02:00,qmHHFoRjxq9fiAKdyTCqxQ,404.8,Fossil Hard coal,11WD4GKM-2CD7--7,intra_zonal,GKM AG Amprion,GKM AG Amprion,1,MAW,PT1M,4,2020-06-13 00:00:00+02:00
3,2017-07-26 14:14:02+02:00,308,DE_LU,Planned maintenance,A03,None,2020-06-13 00:00:00+02:00,3r1LIbc7S59NbqzPyuzAow,404.8,Fossil Hard coal,11WD4GKM-2CD7--7,intra_zonal,GKM AG Amprion,GKM AG Amprion,1,MAW,PT1M,4,2020-06-01 00:00:00+02:00
4,2017-07-26 14:14:03+02:00,595,DE_LU,Planned maintenance,A03,None,2020-06-23 00:00:00+02:00,E-KFelLHPdUCGBMPPmJmZQ,607.1,Fossil Hard coal,11WD4GKM-2CD4--J,intra_zonal,GKM AG TNG,GKM AG TNG,1,MAW,PT1M,3,2020-06-13 00:00:00+02:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35241,2026-10-01 09:09:25+02:00,0,DE_LU,Planned maintenance,A03,None,2026-10-01 09:10:00+02:00,0n-inN6olePdfEgCzGOdwA,608.0,Fossil Gas,11W0-0000-0708-1,intra_zonal,Herne,GUD_HERNE_6,1,MAW,PT1M,8,2026-09-12 01:00:00+02:00
35242,2026-10-01 10:45:41+02:00,0,DE_LU,Unplanned outage,A03,None,2026-10-14 14:30:00+02:00,w5wVLccm7NNignpmyyGO7w,656.0,Fossil Hard coal,11WD7WEIH2S--KWV,Saarland,Weiher,WEIHER_C,1,MAW,PT1M,2,2026-09-15 09:15:00+02:00
35243,2026-10-01 13:28:34+02:00,0,DE_LU,Planned maintenance,A03,None,2026-10-13 18:00:00+02:00,EreVCxJ7yTy48kDbAoMiiA,206.0,Fossil Gas,11WD7LUDW2GA800M,intra_zonal,GuD Ludwigshafen Mitte,Block GT 12,1,MAW,PT1M,7,2026-08-13 08:20:00+02:00
35244,2026-10-01 14:30:26+02:00,0,DE_LU,Planned maintenance,A03,None,2026-10-08 00:00:00+02:00,jCMjBmVsuskmXR9U9Y_mJQ,264.0,Fossil Gas,11W0-0000-0100-I,intra_zonal,GuD Marzahn,GuD Marzahn,1,MAW,PT1M,5,2026-08-22 14:00:00+02:00


In [20]:
units_clean["DE_LU", "production_units_unavailability"] = (
    print_units_unavailability_analysis(
        data=data_entose["DE_LU"]["production_units_unavailability"]
    )
)
units_clean["DE_LU", "production_units_unavailability"]

Initial number of rows: 28259
Duplicated rows: 3557
Duplicated (mrid, revision) rows: 7321

Rows with created_doc_time more than 3 years before start:
Series([], Name: count, dtype: int64)

Rows with created_doc_time on or after end:
businesstype
Planned maintenance    13371
Unplanned outage        1856
Name: count, dtype: int64

Final number of rows: 2154

Planned maintenance:
Number of rows: 1206
created_doc_time range: 2018-08-31 10:47:40+00:00 - 2026-09-16 18:35:45+00:00

Unplanned outage:
Number of rows: 948
created_doc_time range: 2019-12-17 16:30:32+00:00 - 2026-09-12 21:50:39+00:00


,created_doc_time,avail_qty,biddingzone_domain,businesstype,curvetype,docstatus,end,mrid,nominal_power,plant_type,production_resource_id,production_resource_location,production_resource_name,production_resource_psr_name,pstn,qty_uom,resolution,revision,start
0,2018-08-31 12:47:40+02:00,110,DE_LU,Planned maintenance,A03,None,2019-12-13 11:00:00+01:00,o9c5tFNgQZXvNvw0hmLKhA,220.0,Hydro Pumped Storage,11WD2ERZH0002682,intra_zonal,PSW Erzhausen,,1,MAW,PT1M,2,2019-12-13 10:00:00+01:00
1,2019-07-17 13:29:42+02:00,137,DE_LU,Planned maintenance,A03,None,2020-05-19 00:00:00+02:00,_2NZWD5XFra_jHXDi3eTnQ,272.6,Fossil Gas,11WD2FFW-000203E,intra_zonal,HKW West,,17281,MAW,PT1M,1,2020-05-12 00:00:00+02:00
2,2019-07-17 13:29:43+02:00,128,DE_LU,Planned maintenance,A03,None,2020-04-28 00:00:00+02:00,EiLUGf31ZG7LYj1m71s4Tg,272.6,Fossil Gas,11WD2FFW-000203E,intra_zonal,HKW West,,1,MAW,PT1M,1,2020-04-03 00:00:00+02:00
3,2019-08-21 15:34:24+02:00,110,DE_LU,Planned maintenance,A03,None,2020-12-07 11:00:00+01:00,bIp_FMVjDbhUCXPgkKHYjQ,220.0,Hydro Pumped Storage,11WD2ERZH0002682,intra_zonal,PSW Erzhausen,,1,MAW,PT1M,2,2020-12-07 10:00:00+01:00
4,2019-11-05 09:47:44+01:00,157.1,DE_LU,Planned maintenance,A03,None,2019-12-10 00:00:00+01:00,APhRSmpLezdMVefc1ZmGgQ,272.6,Fossil Gas,11WD2FFW-000203E,intra_zonal,HKW West,,1,MAW,PT1M,1,2019-11-05 10:00:00+01:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17373,2026-09-12 00:25:24+02:00,0,DE_LU,Planned maintenance,A03,None,2026-09-15 02:00:00+02:00,TSz7iw0qY-uz7huTPKyXxA,400.0,Wind Offshore,11WD2OGTI000264J,Nordsee AWZ,Global Tech I,,1,MAW,PT1M,1,2026-09-14 02:00:00+02:00
17374,2026-09-12 23:50:39+02:00,31,DE_LU,Unplanned outage,A03,None,2026-09-13 00:00:00+02:00,zBeTzyARDSJoldUm7rpjqA,200.0,Wind Offshore,11W0-0000-0261-N,intra_zonal,Trianel Windpark Borkum II,,1,MAW,PT1M,1,2026-09-12 23:30:00+02:00
17377,2026-09-14 12:39:30+02:00,0,DE_LU,Planned maintenance,A03,None,2026-09-15 12:00:00+02:00,4TJN2jv8Ohb9EpFQNdC3OA,252.0,Wind Offshore,11WD2OGW20003378,intra_zonal,Gode Wind II,,1,MAW,PT1M,1,2026-09-15 07:00:00+02:00
17379,2026-09-16 11:52:46+02:00,143,DE_LU,Planned maintenance,A03,None,2026-09-17 08:00:00+02:00,27rWHBL2VJLHEIUNzxzG3g,315.0,Wind Offshore,11W0-0000-0952-Q,intra_zonal,Windanker,,3031,MAW,PT1M,2,2026-09-16 19:00:00+02:00


**FR - generation/production units unavailability**

In [21]:
units_clean["FR", "generation_units_unavailability"] = (
    print_units_unavailability_analysis(
        data=data_entose["FR"]["generation_units_unavailability"]
    )
)
units_clean["FR", "generation_units_unavailability"]

Initial number of rows: 35117
Duplicated rows: 4707
Duplicated (mrid, revision) rows: 559

Rows with created_doc_time more than 3 years before start:
businesstype
Planned maintenance    2
Name: count, dtype: int64

Rows with created_doc_time on or after end:
businesstype
Planned maintenance    13645
Unplanned outage        7838
Name: count, dtype: int64

Final number of rows: 8366

Planned maintenance:
Number of rows: 6306
created_doc_time range: 2017-05-09 22:28:10+00:00 - 2026-10-01 13:04:38+00:00

Unplanned outage:
Number of rows: 2060
created_doc_time range: 2019-12-01 13:18:35+00:00 - 2026-10-01 13:09:15+00:00


,created_doc_time,avail_qty,biddingzone_domain,businesstype,curvetype,docstatus,end,mrid,nominal_power,plant_type,production_resource_id,production_resource_location,production_resource_name,production_resource_psr_name,pstn,qty_uom,resolution,revision,start
0,2017-05-10 00:28:10+02:00,0,FR,Planned maintenance,A03,None,2020-05-22 00:00:00+02:00,Sqf5vs5E00r6FLRJj5tViA,85.0,Fossil Oil,17W100P100P0260J,FRANCE,BRENNILIS 2,BRENNILIS 2,1,MAW,PT1M,1,2020-05-10 00:00:00+02:00
1,2017-07-27 11:47:42+02:00,0,FR,Planned maintenance,A03,None,2020-06-24 00:00:00+02:00,fQA75hZSyfoRanqP97e8Vg,182.0,Fossil Oil,17W100P100P02691,FRANCE,VAIRES 3,VAIRES 3,1,MAW,PT1M,6,2020-06-06 00:00:00+02:00
2,2017-07-27 11:48:17+02:00,0,FR,Planned maintenance,A03,None,2020-05-15 00:00:00+02:00,T0vaNcErI_LIQzxNrd0H0g,85.0,Fossil Oil,17W100P100P0263D,FRANCE,DIRINON 1,DIRINON 1,1,MAW,PT1M,7,2020-05-03 00:00:00+02:00
3,2017-07-27 14:10:17+02:00,0,FR,Planned maintenance,A03,None,2020-05-21 00:00:00+02:00,ddB4yAEd6LFuAZTFKNNNzQ,185.0,Fossil Oil,17W100P100P0271E,FRANCE,MONTEREAU 6,MONTEREAU 6,1,MAW,PT1M,7,2020-05-02 00:00:00+02:00
4,2017-07-27 14:10:29+02:00,0,FR,Planned maintenance,A03,None,2020-07-06 00:00:00+02:00,EnGXyYo29UTQnfxNw3B5KQ,580.0,Fossil Hard coal,17W100P100P0236G,FRANCE,CORDEMAIS 4,CORDEMAIS 4,1,MAW,PT1M,9,2020-05-30 00:00:00+02:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29846,2026-10-01 12:04:40+02:00,0,FR,Planned maintenance,A03,None,2026-10-03 23:00:00+02:00,gO1p-Zebazg7Dj9l3UJRhw,1300.0,Nuclear,17W100P100P0190E,FRANCE,CATTENOM 4,CATTENOM 4,1,MAW,PT1M,26,2026-01-31 00:00:00+01:00
29847,2026-10-01 14:27:44+02:00,0,FR,Planned maintenance,A03,None,2026-10-01 20:00:00+02:00,odPB6btw9Gnfc54rK_2WOQ,1310.0,Nuclear,17W100P100P0211W,FRANCE,GOLFECH 1,GOLFECH 1,1,MAW,PT1M,27,2026-05-08 23:30:00+02:00
29848,2026-10-01 14:35:43+02:00,0,FR,Planned maintenance,A03,None,2027-03-26 17:00:00+01:00,bIMMKzHPUKpyCawXbyVhuA,182.0,Hydro Water Reservoir,17W100P100P0300X,FRANCE,VILLARODIN,VILLARODIN 1,1,MAW,PT1M,9,2026-04-24 17:00:00+02:00
29849,2026-10-01 15:04:38+02:00,0,FR,Planned maintenance,A03,None,2026-11-04 17:00:00+01:00,f23r-pNCz1rCLGE78cQHjQ,45.0,Hydro Run-of-river and pondage,17W000001017789F,France,CHAUTAGNE,CHAUTAGNE G2,1,MAW,PT1M,7,2026-09-07 07:00:00+02:00


In [22]:
units_clean["FR", "production_units_unavailability"] = (
    print_units_unavailability_analysis(
        data=data_entose["FR"]["production_units_unavailability"]
    )
)
units_clean["FR", "production_units_unavailability"]

Initial number of rows: 7115
Duplicated rows: 515
Duplicated (mrid, revision) rows: 0

Rows with created_doc_time more than 3 years before start:
businesstype
Planned maintenance    5
Name: count, dtype: int64

Rows with created_doc_time on or after end:
businesstype
Planned maintenance    3563
Unplanned outage        737
Name: count, dtype: int64

Final number of rows: 2295

Planned maintenance:
Number of rows: 2080
created_doc_time range: 2017-10-26 06:35:07+00:00 - 2026-09-28 14:42:38+00:00

Unplanned outage:
Number of rows: 215
created_doc_time range: 2019-12-01 14:52:12+00:00 - 2026-09-15 20:38:36+00:00


,created_doc_time,avail_qty,biddingzone_domain,businesstype,curvetype,docstatus,end,mrid,nominal_power,plant_type,production_resource_id,production_resource_location,production_resource_name,production_resource_psr_name,pstn,qty_uom,resolution,revision,start
5,2017-10-26 08:35:07+02:00,0,FR,Planned maintenance,A03,None,2020-07-31 17:00:00+02:00,LoE_udImWn8ysmP9U6DZ3g,808.0,Hydro Pumped Storage,17W100P100P02780,FRANCE,REVIN,,1,MAW,PT1M,12,2020-04-03 07:00:00+02:00
6,2017-12-27 09:27:36+01:00,0,FR,Planned maintenance,A03,None,2020-10-16 17:00:00+02:00,olL92SoEi3EMw3_20xR76A,384.0,Hydro Pumped Storage,17W100P100P02748,FRANCE,COCHE,,1,MAW,PT1M,7,2020-10-05 07:00:00+02:00
7,2019-03-20 16:46:25+01:00,0,FR,Planned maintenance,A03,None,2020-06-18 17:00:00+02:00,ltjxMg2Qs6SPUgGN-FI-_g,366.0,Hydro Water Reservoir,17W100P100P02861,FRANCE,MONTEYNARD,,1,MAW,PT1M,1,2020-06-16 07:00:00+02:00
8,2019-03-27 08:31:34+01:00,142,FR,Planned maintenance,A03,None,2020-06-26 17:00:00+02:00,Pr1GryBeRaZXPlMgtGRl-w,285.0,Hydro Water Reservoir,17W100P100P0301V,FRANCE,VOUGLANS,,1,MAW,PT1M,1,2020-06-15 07:00:00+02:00
9,2019-03-27 10:34:12+01:00,0,FR,Planned maintenance,A03,None,2020-02-05 01:00:00+01:00,mutrHD_1pB3tuzyhD5wFBA,240.0,Hydro Water Reservoir,17W100P100P02942,FRANCE,SISTERON,,1,MAW,PT1M,1,2020-02-04 12:00:00+01:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6591,2026-09-15 14:03:22+02:00,0,FR,Unplanned outage,A03,None,2026-09-15 22:00:00+02:00,gUZ1TrSrS2_nOnIPsDbQyQ,435.0,Fossil Gas,17W100P100P0345B,FRANCE,SAINT AVOLD 8,,1,MAW,PT1M,2,2026-09-15 06:18:00+02:00
6592,2026-09-15 22:38:36+02:00,0,FR,Unplanned outage,A03,None,2026-09-16 07:00:00+02:00,A7fj2QVixamtzsxH0KZg6w,435.0,Fossil Gas,17W100P100P0345B,FRANCE,SAINT AVOLD 8,,1,MAW,PT1M,1,2026-09-15 22:37:00+02:00
6595,2026-09-24 14:39:45+02:00,218,FR,Planned maintenance,A03,None,2026-09-25 17:00:00+02:00,hGpxKhx4DrBiB1NhbYOrHw,384.0,Hydro Water Reservoir,17W100P100P02926,FRANCE,SERRE PONCON,,1,MAW,PT1M,2,2026-08-28 17:00:00+02:00
6596,2026-09-24 17:34:04+02:00,0,FR,Planned maintenance,A03,None,2026-10-20 23:59:00+02:00,krOoHsJMfu9uUojDnlkwVg,435.0,Fossil Gas,17W100P100P0344D,FRANCE,SAINT AVOLD 7,,1,MAW,PT1M,1,2026-07-15 00:00:00+02:00


**Unavailability records by outage start month (after deduplication and filtering)**

In [23]:
for bz in ["DE_LU", "FR"]:
    for name in ["generation_units_unavailability", "production_units_unavailability"]:
        start = units_clean[bz, name]["start"]
        start = start[start >= "2023-01-01"]
        print(f"{bz} - {name} (number of records by outage start month)")
        print(
            pd.crosstab(
                index=start.dt.year.to_numpy(),
                columns=start.dt.month.to_numpy(),
                rownames=["year"],
                colnames=["month"],
            ),
            end="\n\n",
        )

DE_LU - generation_units_unavailability (number of records by outage start month)
month   1    2    3    4    5    6    7    8    9   10   11   12
year                                                            
2023     1    3    4    0    9    0    2    0    0   1    0    0
2024     1    1    0    0    0    0    1    0    0   0    0    1
2025     3    2    1    2    1    0    1    1    4  23  156  194
2026   230  237  335  270  192  277  346  439  199   0    0    0

DE_LU - production_units_unavailability (number of records by outage start month)
month  1   2   3   4   5   6   7    8   9   11  12
year                                              
2025    0   0   0   0   0   0   0    0   0  51  54
2026   80  55  48  43  19  73  67  128  30   0   0

FR - generation_units_unavailability (number of records by outage start month)
month   1    2    3    4    5    6    7    8    9   10   11   12
year                                                            
2023     2    0    3    9    8 

In [24]:
for bz, datasets_bz in data_entose.items():
    print(f"\n--- {bz} " + "-" * 40)
    for name, data in datasets_bz.items():
        find_missing_dates(data=data, name=name, freq="h")


--- AT ----------------------------------------

--- [load_forecast] Missing timestamps (freq=h) ---
Count: 0

--- [day_ahead_prices] Missing timestamps (freq=h) ---
Count: 3

Missing ranges:
  2023-11-30 00:00:00+01:00 (1 timestamp)
  2024-11-30 00:00:00+01:00 (1 timestamp)
  2025-11-30 00:00:00+01:00 (1 timestamp)

Missing timestamps by day of week:
  Thursday: 1
  Saturday: 1
  Sunday: 1

--- [generation_wind_solar_forecast] Missing timestamps (freq=h) ---
Count: 0

--- [generation_forecast] Missing timestamps (freq=h) ---
Count: 0

--- CZ ----------------------------------------

--- [load_forecast] Missing timestamps (freq=h) ---
Count: 0

--- [day_ahead_prices] Missing timestamps (freq=h) ---
Count: 3

Missing ranges:
  2023-11-30 00:00:00+01:00 (1 timestamp)
  2024-11-30 00:00:00+01:00 (1 timestamp)
  2025-11-30 00:00:00+01:00 (1 timestamp)

Missing timestamps by day of week:
  Thursday: 1
  Saturday: 1
  Sunday: 1

--- [generation_wind_solar_forecast] Missing timestamps (freq=

### 2.4 Index Integrity – Observations

Resolution and switch timing differ by zone and dataset; a common UTC grid is required before any cross-zone merge. `day_ahead_prices` is the one exception with an identical structure across all non-PL zones.

The 00:00 timestamp is missing on 30 November (2023–2025) in every non-PL zone at the same UTC instant, pointing to a systematic cause (download or publication window) rather than zone-specific outages — a single, predictable gap to interpolate. LT `generation_forecast` is missing 290 hourly timestamps in 5 gaps, the longest lasting 8 days (193 hours), and stays excluded. DE_LU `load_forecast` and `generation_forecast` are missing the first hour of the autumn DST-change Sundays, as in the wind/solar forecast. Whole-day gaps remain in FR (`generation_wind_solar_forecast`, `generation_forecast`), SK (`generation_forecast`) and SE_4 (`load_forecast`, `generation_wind_solar_forecast`, `generation_forecast`; the longest lasts 2 days). FR `generation_forecast`, LT `load_forecast` and LT `generation_wind_solar_forecast` additionally contain isolated gaps of up to 1 hour.

**Unavailability tables (DE_LU, FR).** The pattern is the same as in PL. After removing duplicates and retrospective records, `generation_units_unavailability` shrinks from 116,263 to 9,886 rows (DE_LU) and from 35,117 to 8,366 (FR), and `production_units_unavailability` from 28,259 to 2,154 (DE_LU) and from 7,115 to 2,295 (FR). By outage start month, the generation-unit tables have at most 15 records per month from 2023 to September 2025 and 156–457 per month from November 2025. DE_LU production has no records before November 2025, and FR production has at most 11 per month in 2023 and only 3 records between 2024 and September 2025. Both tables are therefore excluded in DE_LU and FR as well (decision in section 1.4).

## 3. Conclusions and Data Cleaning / Preprocessing Decisions

**Main findings**
PL core series have no missing timestamps; remaining NaN values are structural or DST-related. `generation_units_unavailability` and `production_units_unavailability` are excluded in all zones: most records were published only after the outage ended, and usable history starts in autumn 2025 (sections 1.4 and 2.4). LT `generation_forecast` is unusable.

### 3.1 Data Cleaning Decisions

**Poland (PL)**

* Drop `Other`, `Other renewable`, `Hydro Pumped Storage - Actual Consumption` and `Wind Offshore` from `generation`; drop `Wind Offshore` from `generation_wind_solar_forecast`.
* Deduplicate `day_ahead_prices`, keeping the first record.
* Drop `generation_units_unavailability` and `production_units_unavailability` (section 1.4).

**Germany (DE_LU)**

* Drop `generation_units_unavailability` and `production_units_unavailability` (section 2.4).

**France (FR)**

* Drop `Wind Offshore` from `generation_wind_solar_forecast`.
* Drop `generation_units_unavailability` and `production_units_unavailability` (section 2.4).

**Lithuania (LT)**

* Exclude `generation_forecast` entirely, as the series is not usable.

### 3.2 Data Preprocessing Decisions

**Time Index**

* Convert all relevant time indices to UTC before merging, comparing or aggregating data across bidding zones.
* Perform `day_ahead_prices` deduplication after conversion to UTC.

**Poland (PL)**

* No interpolation is needed for the PL core series — no missing timestamps. Remaining NaN values are treated as structural/DST-related, not as gaps to impute.

**Other bidding zones**

* Gaps of up to 3 hours – linear interpolation on the UTC grid: the missing 30 November 00:00 price timestamp in all non-PL zones, DST-related gaps in DE_LU/FR and isolated gaps in FR `generation_forecast`, LT `load_forecast` and LT `generation_wind_solar_forecast`.
* Longer gaps (more than 3 hours) in weather-driven series (`generation_wind_solar_forecast`) – copy the same hours from the previous available day (D-1): FR `Solar`/`Wind Onshore` (2026-01-02), HU `Wind Onshore` (2026-02-16, 15:30–23:45), the whole-day gap in FR (2023-04-18) and the five whole-day gaps in SE_4 (for the 2-day gap use the last available day before it).
* Whole-day gaps in demand- and schedule-driven series – copy the same hours from one week earlier (D-7): SE_4 `load_forecast` (2025-12-04), `generation_forecast` in FR (2025-03-06), SK (2023-12-08) and SE_4 (2025-12-02).
* D-1 is used for weather-driven series because weather persists from one day to the next, and D-7 for demand- and schedule-driven series because they follow a weekly (weekday/weekend) cycle. The hours are matched by local time of day. The 3-hour limit keeps interpolation within a small part of the daily cycle (solar, load); the gaps found are either up to 1 hour or at least 8.5 hours long, so no case lies close to the limit.
* Reservoir data (AT, NO, SE): forward fill only, including the 14-day step.